In [ ]:
# Set up the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64

# JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME ##### 
#### DONE ####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
try:
    from CRUD_Python_Module_Enhanced import AnimalShelter
except ImportError:
    try:
        from CRUD_Python_Module_Enhanced import AnimalShelter
    except ImportError:
        try:
            from CRUD_Python_Module_Enhanced import AnimalShelter
        except ImportError as exc:
            raise ImportError(
                "Could not import AnimalShelter. Ensure the module exists in the notebook's directory "
                "and is named either 'CRUD_Python_Module.py' or 'animal_shelter.py'."
            ) from exc

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

#### DONE ####

username = "aacuser"
password = "messieJessie"

# Connect to database via CRUD Module 
# wasn't compatible wiht CrUD_Python_Module.py - so I commented it out and used the default constructor instead
# db = AnimalShelter(username, password)

db = AnimalShelter()

# class read method must support return of list object and accept projection JSON input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invalid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will return a new dataframe that does not contain the dropped column(s)
# df.drop(columns=['_id'],inplace=True) replaced with the following code to check if the column exists before dropping it
df.drop(columns=['_id'], inplace=True, errors='ignore')
## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# -----------------------------
# DASHBOARD LAYOUT / VIEW
# -----------------------------
#image_filename = 'Grazioso Salvare Logo.png' code corrected below
image_filename = 'Grazioso Salvare Logo Project 2.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
    
    
    # -----------------------------
    # Dashboard Title
    # -----------------------------
    html.Center(html.B(html.H1('CS-340 Dashboard'))),

    # -----------------------------
    # Logo + Unique Identifier
    # -----------------------------
    html.Div([ 
        html.Img( 
            src='data:image/png;base64,{}'.format(encoded_image.decode()),
            style={'height': '120px'}
        ),

        # Required unique identifier (your name)
        html.H4("Dashboard by Jessica Puffenbarger")
    ], style={'textAlign': 'center'}),

    html.Hr(),

    # -----------------------------
    # Interactive Filtering Options
    # -----------------------------
    html.Div([
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'All', 'value': 'all'},
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain/Wilderness Rescue', 'value': 'mountain'},
                {'label': 'Disaster/Individual Tracking', 'value': 'disaster'}
            ],
            value='all',  # Default selection
            inline=True,
            style={'fontSize': '18px', 'padding': '10px'}
        ),
        html.Button( 
            "Reset", 
            id="reset-button", 
            n_clicks=0, 
            style={'margin': '10px', 'fontSize': '16px'} 
        )
    ]),

    html.Hr(),

    # -----------------------------
    # Interactive Data Table
    # -----------------------------
    dash_table.DataTable(
        id='datatable-id',

        # Table columns based on MongoDB fields
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True}
                 for i in df.columns],

        # Initial unfiltered data
        data=df.to_dict('records'),

        # User-friendly features
        page_size=10,
        sort_action='native',
        filter_action='native',
        row_selectable='single',
        selected_rows=[],
        style_table={'overflowX': 'auto'},
        style_cell={'textAlign': 'left', 'padding': '5px'},
        style_header={'backgroundColor': 'lightgrey', 'fontWeight': 'bold'}
    ),

    html.Br(),
    html.Hr(),

    # -----------------------------
    # Side-by-Side Charts
    # -----------------------------
    html.Div(
        className='row',
        style={'display': 'flex'},
        children=[
            # Chart container
            html.Div(
                id='graph-id',
                className='col s12 m6'
            ),

            # Map container
            html.Div(
                id='map-id',
                className='col s12 m6'
            )
        ]
    )
])


    
#############################################
# Interaction Between Components / Controller
#############################################

# -----------------------------
# Filter the Data Table
# -----------------------------
@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    """
    Updates the data table based on the selected rescue filter.
    Uses MongoDB queries through the CRUD module.
    """

    # Enhancement 1: Create a Rescue Filter Dictionary
    RESCUE_BREEDS = {
        "water": [
            "Labrador Retriever Mix",
            "Chesapeake Bay Retriever",
            "Newfoundland"
        ],
        "mountain": [
            "German Shepherd",
            "Belgian Malinois",
            "Border Collie"
        ],
        "disaster": [
            "Bloodhound",
            "Coonhound",
            "Beagle"
        ]
    }

    if filter_type in RESCUE_BREEDS:
        query = {"breed": {"$in": RESCUE_BREEDS[filter_type]}}
    else:
        query = {}

    # Query MongoDB with error handling
    try:
        results = db.read(query)

        # Check if query returned data
        if not results:
            return []

        dff = pd.DataFrame.from_records(results)

        # Remove MongoDB _id field
        if '_id' in dff.columns:
            dff.drop(columns=['_id'], inplace=True)

        return dff.to_dict('records')

    except Exception as e:
        print(f"Database Error: {e}")
        return []

# -----------------------------
# Update the Pie Chart
# -----------------------------
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)

#Handle Empty Results
def update_graphs(viewData):

    if viewData is None:
        dff = df
    else:
        dff = pd.DataFrame.from_dict(viewData)

    #Added Empty Dataset Handling
    # Check for empty data
    if dff.empty:
        return [html.H4("No matching records found")]

    # Count the number of animals in each breed
    breed_counts = dff['breed'].value_counts().reset_index()

    breed_counts.columns = ['breed', 'count']

    fig = px.bar(
        breed_counts,
        x='breed',
        y='count',
        title='Breed Frequency Analysis'
    )

    return [dcc.Graph(figure=fig)]


# -----------------------------
# Highlight Selected Column
# -----------------------------
@app.callback( 
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if not selected_columns:
        return [] # no highlight when nothing is selected
    
    return [{
        'if': {'column_id': col},
        'background_color': '#D2F3FF'
        } for col in selected_columns]


# -----------------------------
# Update the Map Based on Row Selection
# -----------------------------
@app.callback(
    Output('map-id', "children"),
    [
        Input('datatable-id', "derived_virtual_data"),
        Input('datatable-id', "derived_virtual_selected_rows")
    ]
)
def update_map(viewData, index):
    """
    Updates the geolocation map based on the selected row in the data table.
    """

    if viewData is None or index is None:
        return

    dff = pd.DataFrame.from_dict(viewData)

    # Only one row can be selected → extract index
    row = index[0] if index else 0
   
    # Retrieve animal information using column names
    latitude = dff.iloc[row]["location_lat"]
    longitude = dff.iloc[row]["location_long"]
    breed = dff.iloc[row]["breed"]
    name = dff.iloc[row]["name"] 

    # Build map
    return [
        dl.Map(
            style={'width': '1000px', 'height': '500px'},
            center=[30.75, -97.48],  # Austin, TX
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
            
                #Improve Map Data Access
                # Marker for selected animal
                dl.Marker(
                    position=[latitude, longitude],
                    children=[
                        dl.Tooltip(breed),  # Breed
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.H3(name),
                            html.P(f"Breed: {breed}")
                        ])
                    ]
                )
            ]
        )
    ]


# -----------------------------
# Run the App
# -----------------------------
# app.run_server()
app.run()

Error reading documents: localhost:27017: [WinError 10061] No connection could be made because the target machine actively refused it (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 30s, Topology Description: <TopologyDescription id: 6ab9b23576c91c9692c502e2, topology_type: Unknown, servers: [<ServerDescription ('localhost', 27017) server_type: Unknown, rtt: None, error=AutoReconnect('localhost:27017: [WinError 10061] No connection could be made because the target machine actively refused it (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms)')>]>


C:\Users\jessi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\jupyter_dash\jupyter_app.py:103: UserWarning: JupyterDash is deprecated, use Dash instead.
See https://dash.plotly.com/dash-in-jupyter for more details.
  super(JupyterDash, self).__init__(name=name, **kwargs)
